# ML 모델 일반화(Generalization) 실험

## 노트북 목적
이 노트북은 학습된 ML 모델이 **새로운 맥락의 데이터에서도 유효한 성능을 유지하는지**(일반화 능력) 검증합니다.
두 가지 관점에서 일반화를 테스트합니다:

### Part 1: 문항 조합 일반화 (Question Combo)
- 8개 문항(Q1~Q8) 중 일부로 학습하고, 나머지 문항의 피드백을 예측
- 254개 문항 조합 (1문항~7문항 학습) x 24개 알고리즘 = **6,096회 실험**
- 목적: 특정 문항에서 학습한 모델이 다른 문항에도 적용 가능한지 확인

### Part 2: 학생 분할 일반화 (Student Split)
- 학생 단위로 학습/테스트를 분할하여 새로운 학생의 피드백 예측 성능 확인
- 5개 비율(80:20 ~ 40:60) x 3개 seed x 24개 알고리즘 = **360회 실험**
- 목적: 학습 데이터 비율에 따른 성능 변화 분석

### 핵심 기술
- **병렬 처리(Parallel Processing)**: `joblib.Parallel`을 사용하여 24개 알고리즘을 **동시에** 실행
- **Auto-tune**: 최적 병렬 작업 수(n_jobs)를 자동으로 탐색
- **토큰 캐싱**: 형태소 분석 결과를 pickle 파일로 캐시하여 재실행 시 시간 절약


# ML Generalization Experiment (Parallel24, Part1+Part2)

- 원본 `00_ml_generalization_experiment.ipynb`는 수정하지 않습니다.
- 이 노트북은 **Part 1(문항 조합)**과 **Part 2(학생 분할)** 모두에서,
  각 데이터 split 내부의 24개 알고리즘을 `joblib.Parallel`로 동시 실행합니다.
- 중첩 병렬 방지를 위해 모델 내부 `n_jobs`는 1로 고정합니다.


## 1. 환경 설정 및 라이브러리 로드
병렬 처리 시 스레드 충돌을 방지하기 위해 **환경 변수로 내부 스레드 수를 1로 제한**합니다.
이는 외부 병렬(joblib)과 내부 병렬(numpy/scipy)이 동시에 작동하여 성능이 저하되는 것을 방지합니다.

In [11]:

import os
os.environ.setdefault('OMP_NUM_THREADS', '1')  # OpenMP 스레드 수 제한
os.environ.setdefault('MKL_NUM_THREADS', '1')  # Intel MKL 스레드 수 제한
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')

import json
import time
import warnings
from datetime import datetime
from copy import deepcopy
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from joblib import Parallel, delayed
from scipy import stats

from kiwipiepy import Kiwi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

warnings.filterwarnings('ignore')

print(f"실험 시작: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print('라이브러리 로드 완료')


실험 시작: 2026-03-04 00:47:47
라이브러리 로드 완료


## 2. 실험 설정
실험의 모든 하이퍼파라미터와 데이터 경로를 한 곳에서 관리합니다.

**주요 설정:**
- `Q_MAP`: 8개 문항을 양식(a/b)과 문항번호(q1~q4)로 매핑
- `all_combos`: 8개 문항에서 1~7개를 선택하는 모든 조합 = 254가지
- `SPLIT_RATIOS`: 학생 분할 비율 (80:20부터 40:60까지)
- `PARALLEL_N_JOBS`: 동시 실행할 작업 수 (CPU 코어 수 기반 자동 설정)
- `INTERNAL_MODEL_N_JOBS = 1`: 개별 모델 내부 병렬은 비활성화 (중첩 병렬 방지)

In [12]:

# =========================
# Configuration
# =========================
DATA_PATH = Path('../data/feedback_data.xlsx')
RANDOM_SEED = 42
N_FOLDS = 10

SPLIT_RATIOS = [(80, 20), (70, 30), (60, 40), (50, 50), (40, 60)]
SPLIT_SEEDS = [42, 123, 456]

Q_MAP = {
    'Q1': ('a', 'q1'), 'Q2': ('a', 'q2'), 'Q3': ('a', 'q3'), 'Q4': ('a', 'q4'),
    'Q5': ('b', 'q1'), 'Q6': ('b', 'q2'), 'Q7': ('b', 'q3'), 'Q8': ('b', 'q4'),
}
ALL_QUESTIONS = list(Q_MAP.keys())
all_combos = []
for k in range(1, 8):  # 1문항~7문항 조합 생성 (8문항 전부는 테스트 데이터 없음)
    for combo in combinations(ALL_QUESTIONS, k):
        all_combos.append(combo)

CPU_LOGICAL = os.cpu_count() or 1
PARALLEL_BACKEND = 'threading'  # 스레드 기반 병렬 (GIL 해제되는 sklearn에 적합)
INTERNAL_MODEL_N_JOBS = 1  # 모델 내부 병렬 비활성화 (중첩 병렬 방지)

# 기본값: 코어 1개 여유
PARALLEL_N_JOBS = max(1, min(24, CPU_LOGICAL - 1))

AUTO_TUNE_N_JOBS = True
AUTOTUNE_FOLDS = 5
AUTOTUNE_SPEC_LIMIT = 12
N_JOBS_CANDIDATES = sorted(set([
    max(1, CPU_LOGICAL // 2),
    max(1, int(round(CPU_LOGICAL * 0.75))),
    max(1, CPU_LOGICAL - 2),
    max(1, CPU_LOGICAL - 1),
    CPU_LOGICAL,
    24,
]))

CACHE_DIR = Path('./split_files')
CACHE_DIR.mkdir(exist_ok=True)
TOKENIZED_CACHE_PATH = CACHE_DIR / 'tokenized_feedback.pkl'

QC_OUTPUT_DIR = Path('./question_combo_results_parallel24')
QC_PRED_DIR = QC_OUTPUT_DIR / 'predictions'
QC_OUTPUT_DIR.mkdir(exist_ok=True)
QC_PRED_DIR.mkdir(exist_ok=True)

SS_OUTPUT_DIR = Path('./student_split_results_parallel24')
SS_PRED_DIR = SS_OUTPUT_DIR / 'predictions'
SS_OUTPUT_DIR.mkdir(exist_ok=True)
SS_PRED_DIR.mkdir(exist_ok=True)

print('CPU logical cores:', CPU_LOGICAL)
print('Initial parallel jobs:', PARALLEL_N_JOBS)
print('n_jobs candidates:', N_JOBS_CANDIDATES)
print('Parallel backend:', PARALLEL_BACKEND)
print('Model internal n_jobs:', INTERNAL_MODEL_N_JOBS)
print('Question combos:', len(all_combos))


CPU logical cores: 12
Initial parallel jobs: 11
n_jobs candidates: [6, 9, 10, 11, 12, 24]
Parallel backend: threading
Model internal n_jobs: 1
Question combos: 254


## 3. 24개 알고리즘 정의
9개 단일 모델 + 5개 앙상블 조합 x 3개 전략 = **24개 알고리즘**을 정의합니다.

**중요:** 모든 모델의 `n_jobs=1`로 설정하여 개별 모델은 단일 스레드로 실행됩니다.
대신 24개 모델을 `joblib.Parallel`로 **동시에 병렬 실행**합니다.
이 방식이 모델 내부 병렬보다 전체 처리 시간이 더 빠릅니다.

In [13]:

# =========================
# Model definitions (24 algorithms)
# =========================
INDIVIDUAL_MODELS = ['LR', 'SVM-L', 'SVM-RBF', 'CNB', 'DT', 'RF', 'GB', 'XGB', 'LGBM']

ENSEMBLE_COMBOS = {
    'C1_Optimal5': ['SVM-L', 'CNB', 'DT', 'GB', 'XGB'],
    'C2_Alt5':     ['LR', 'SVM-L', 'DT', 'GB', 'XGB'],
    'C3_Optimal3': ['LR', 'SVM-RBF', 'XGB'],
    'C4_Prev4':    ['LR', 'SVM-L', 'CNB', 'RF'],
    'C5_Prev5':    ['LR', 'SVM-RBF', 'CNB', 'RF', 'LGBM'],
}
ENSEMBLE_METHODS = ['Hard Voting', 'Soft Voting', 'Stacking']


def build_algorithm_specs():
    specs = []

    for model_name in INDIVIDUAL_MODELS:
        specs.append({
            'Type': 'Individual',
            'Model': model_name,
            'Combo_Name': None,
            'Method': None,
            'pred_col': f'{model_name}_pred'
        })

    for combo_name, combo_models in ENSEMBLE_COMBOS.items():
        for method in ENSEMBLE_METHODS:
            exp_name = f"{combo_name}_{method.replace(' ', '')}"
            specs.append({
                'Type': 'Ensemble',
                'Model': exp_name,
                'Combo_Name': combo_name,
                'Method': method,
                'combo_models': combo_models,
                'pred_col': exp_name
            })

    return specs


def create_vectorizer():
    return TfidfVectorizer(
        tokenizer=lambda x: x.split() if x else [],
        preprocessor=lambda x: x,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
    )


def get_single_classifier(name, random_state=42, n_jobs=1):
    clfs = {
        'LR': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=random_state, n_jobs=n_jobs),
        'SVM-L': LinearSVC(class_weight='balanced', max_iter=2000, random_state=random_state),
        'SVM-RBF': SVC(kernel='rbf', class_weight='balanced', random_state=random_state, probability=True),
        'CNB': ComplementNB(alpha=1.0),
        'DT': DecisionTreeClassifier(class_weight='balanced', random_state=random_state),
        'RF': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=random_state, n_jobs=n_jobs),
        'GB': GradientBoostingClassifier(n_estimators=100, random_state=random_state),
        'XGB': XGBClassifier(n_estimators=100, eval_metric='mlogloss', random_state=random_state, n_jobs=n_jobs, verbosity=0),
        'LGBM': LGBMClassifier(n_estimators=100, class_weight='balanced', random_state=random_state, n_jobs=n_jobs, verbose=-1),
    }
    return clfs[name]


def get_soft_classifier(name, random_state=42, n_jobs=1):
    if name == 'SVM-L':
        return SVC(kernel='linear', class_weight='balanced', random_state=random_state, probability=True)
    return get_single_classifier(name, random_state, n_jobs=n_jobs)


def build_ensemble(combo_models, method, random_state=42, n_jobs=1):
    if method == 'Hard Voting':
        estimators = [(n.lower().replace('-', '_'), get_single_classifier(n, random_state, n_jobs=n_jobs)) for n in combo_models]
        return VotingClassifier(estimators=estimators, voting='hard', n_jobs=n_jobs)

    if method == 'Soft Voting':
        estimators = [(n.lower().replace('-', '_'), get_soft_classifier(n, random_state, n_jobs=n_jobs)) for n in combo_models]
        return VotingClassifier(estimators=estimators, voting='soft', n_jobs=n_jobs)

    if method == 'Stacking':
        estimators = [(n.lower().replace('-', '_'), get_soft_classifier(n, random_state, n_jobs=n_jobs)) for n in combo_models]
        return StackingClassifier(
            estimators=estimators,
            final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000, random_state=random_state, n_jobs=n_jobs),
            cv=5,
            n_jobs=n_jobs,
        )

    raise ValueError(f'Unknown method: {method}')


specs = build_algorithm_specs()
print(f'총 알고리즘: {len(specs)}개')


총 알고리즘: 24개


## 4. 평가 유틸리티 함수
- **`run_cv_and_predict_test()`**: 10-Fold CV를 수행하고, 각 fold 모델의 테스트 예측을 다수결로 결합
- **`run_one_algorithm()`**: 하나의 알고리즘에 대해 CV + 테스트 평가를 수행 (병렬 실행의 단위 작업)
- **`run_algorithms_parallel()`**: 24개 알고리즘을 `joblib.Parallel`로 동시 실행

> **다수결 예측(Majority Voting from CV)**: 10개 fold에서 학습된 10개 모델이 각각 테스트 세트를 예측한 후, 샘플별로 가장 많이 예측된 클래스를 최종 예측으로 사용합니다.

In [14]:

# =========================
# Evaluation utilities
# =========================
def run_cv_and_predict_test(clf, X_train, y_train, X_test, n_splits=10, random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    cv_scores = {'acc': [], 'f1': [], 'kappa': []}
    test_predictions = []

    for tr_idx, val_idx in skf.split(X_train, y_train):
        clf_fold = deepcopy(clf)
        clf_fold.fit(X_train[tr_idx], y_train[tr_idx])

        y_val_pred = clf_fold.predict(X_train[val_idx])
        cv_scores['acc'].append(accuracy_score(y_train[val_idx], y_val_pred))
        cv_scores['f1'].append(f1_score(y_train[val_idx], y_val_pred, average='macro'))
        cv_scores['kappa'].append(cohen_kappa_score(y_train[val_idx], y_val_pred))

        test_predictions.append(clf_fold.predict(X_test))

    test_preds_array = np.array(test_predictions)
    final_test_pred = stats.mode(test_preds_array, axis=0, keepdims=False)[0]

    cv_result = {
        'CV_Acc': float(np.mean(cv_scores['acc'])),
        'CV_Acc_Std': float(np.std(cv_scores['acc'])),
        'CV_F1': float(np.mean(cv_scores['f1'])),
        'CV_F1_Std': float(np.std(cv_scores['f1'])),
        'CV_Kappa': float(np.mean(cv_scores['kappa'])),
        'CV_Kappa_Std': float(np.std(cv_scores['kappa'])),
    }
    return cv_result, final_test_pred


def evaluate_test(y_true, y_pred):
    return {
        'Test_Acc': accuracy_score(y_true, y_pred),
        'Test_F1': f1_score(y_true, y_pred, average='macro'),
        'Test_F1_W': f1_score(y_true, y_pred, average='weighted'),
        'Test_Kappa': cohen_kappa_score(y_true, y_pred),
    }


def run_one_algorithm(spec, X_train, y_train, X_test, y_test, context_meta, n_splits=None):
    model_name = spec['Model']
    n_splits = n_splits or N_FOLDS

    try:
        if spec['Type'] == 'Individual':
            clf = get_single_classifier(model_name, RANDOM_SEED, n_jobs=INTERNAL_MODEL_N_JOBS)
        else:
            clf = build_ensemble(
                spec['combo_models'],
                spec['Method'],
                RANDOM_SEED,
                n_jobs=INTERNAL_MODEL_N_JOBS,
            )

        cv_result, y_pred = run_cv_and_predict_test(
            clf,
            X_train,
            y_train,
            X_test,
            n_splits=n_splits,
            random_state=RANDOM_SEED,
        )
        test_metrics = evaluate_test(y_test, y_pred)

        row = {
            **context_meta,
            'Type': spec['Type'],
            'Model': model_name,
            'Combo_Name': spec['Combo_Name'],
            'Method': spec['Method'],
            **cv_result,
            **test_metrics,
        }
        return {'ok': True, 'row': row, 'pred_col': spec['pred_col'], 'y_pred': y_pred, 'model': model_name}

    except Exception as e:
        return {'ok': False, 'error': str(e), 'pred_col': spec['pred_col'], 'model': model_name}


def run_algorithms_serial(specs, X_train, y_train, X_test, y_test, context_meta, n_splits=None):
    return [
        run_one_algorithm(spec, X_train, y_train, X_test, y_test, context_meta, n_splits=n_splits)
        for spec in specs
    ]


def run_algorithms_parallel(specs, X_train, y_train, X_test, y_test, context_meta, n_jobs=None, backend='threading', n_splits=None):
    n_jobs = n_jobs or PARALLEL_N_JOBS
    return Parallel(n_jobs=n_jobs, backend=backend, batch_size=1, verbose=0)(  # 24개 알고리즘 동시 실행
        delayed(run_one_algorithm)(spec, X_train, y_train, X_test, y_test, context_meta, n_splits=n_splits)
        for spec in specs
    )


## 5. 데이터 로드 및 토큰화
- 전체 2,080개 피드백 데이터를 로드하고, 학생/양식/문항 정보를 ID에서 추출합니다.
- 형태소 분석 결과를 **pickle 캐시 파일**로 저장하여, 재실행 시 분석 시간을 절약합니다.
- ID 형식 예: `01_aq1_03` → 학생 01, 양식 a, 문항 q1, 피드백 03

In [ ]:

# =========================
# Data load + tokenization cache
# =========================
class KiwiLemmaTokenizer:
    INCLUDE_POS = {'NNG', 'NNP', 'NNB', 'NR', 'VV', 'VA', 'VX', 'MAG', 'MAJ'}

    def __init__(self):
        self.kiwi = Kiwi()

    def tokenize_with_lemma(self, text):
        if pd.isna(text) or str(text).strip() == '':
            return ''
        tokens = self.kiwi.tokenize(str(text))
        return ' '.join(t.lemma or t.form for t in tokens if t.tag in self.INCLUDE_POS)


def enrich_id_columns(df_in):
    df_out = df_in.copy()
    df_out['student'] = df_out['id'].astype(str).str.extract(r'^(\d+)_')
    df_out['form'] = df_out['id'].astype(str).str.extract(r'_([ab])q')
    df_out['question'] = df_out['id'].astype(str).str.extract(r'_[ab](q\d)_')
    return df_out


def is_valid_token_cache(df_cache):
    required = {'id', 'feedback_text', 'label', 'student', 'form', 'question', 'tokenized'}
    if not required.issubset(df_cache.columns):
        return False

    student_na_ratio = df_cache['student'].isna().mean()
    nonempty_tokens = (df_cache['tokenized'].fillna('').astype(str).str.strip() != '').sum()

    if student_na_ratio > 0.01:
        return False
    if nonempty_tokens == 0:
        return False
    return True


raw_df = pd.read_excel(DATA_PATH)
raw_df = enrich_id_columns(raw_df)

cache_loaded = False  # 캐시 파일이 유효하지 않으면 재생성
if TOKENIZED_CACHE_PATH.exists():
    try:
        cached_df = pd.read_pickle(TOKENIZED_CACHE_PATH)

        if is_valid_token_cache(cached_df):
            df = cached_df.copy()
            cache_loaded = True
            print(f'토큰 캐시 로드: {TOKENIZED_CACHE_PATH} ({len(df)}행)')
        else:
            print('기존 토큰 캐시 무효(컬럼/값 문제). 캐시 재생성 진행...')

    except Exception as e:
        print('기존 토큰 캐시 로드 실패(버전 호환/파일 손상 가능). 캐시 재생성 진행...')
        print(f'  cache load error: {type(e).__name__}: {e}')

if not cache_loaded:
    df = raw_df.copy()
    print('토큰화 시작 (최초 1회 또는 캐시 재생성)...')
    tokenizer = KiwiLemmaTokenizer()
    df['tokenized'] = df['feedback_text'].apply(tokenizer.tokenize_with_lemma)
    df.to_pickle(TOKENIZED_CACHE_PATH)
    print(f'토큰 캐시 저장: {TOKENIZED_CACHE_PATH}')

label_encoder = LabelEncoder()
label_encoder.fit(df['label'])

print(f"데이터 준비 완료: {len(df)}행, 학생 {df['student'].nunique()}명")
print(df['label'].value_counts().sort_index())


## 6. 데이터 분할 함수
두 가지 분할 방식을 정의합니다:
- **`split_by_questions()`**: 문항 단위 분할 - 특정 문항의 피드백으로 학습, 나머지 문항으로 테스트
- **`split_by_students()`**: 학생 단위 분할 - 특정 학생들의 피드백으로 학습, 나머지 학생으로 테스트
- **`prepare_features()`**: TF-IDF 벡터화 수행

In [16]:

# =========================
# Split helpers
# =========================
def get_question_data(df_in, question_label):
    form, q = Q_MAP[question_label]
    return df_in[(df_in['form'] == form) & (df_in['question'] == q)]


def split_by_questions(df_in, train_questions):
    test_questions = [q for q in ALL_QUESTIONS if q not in train_questions]
    train_df = pd.concat([get_question_data(df_in, q) for q in train_questions], ignore_index=True)
    test_df = pd.concat([get_question_data(df_in, q) for q in test_questions], ignore_index=True)
    return train_df, test_df


def split_by_students(df_in, train_ratio, seed):
    student_ids = sorted(df_in['student'].unique())
    n_students = len(student_ids)
    n_train = int(n_students * train_ratio / 100)

    rng = np.random.RandomState(seed)  # 재현 가능한 랜덤 분할
    shuffled = rng.permutation(student_ids)  # 학생 ID를 랜덤하게 섞기

    train_students = set(shuffled[:n_train])
    test_students = set(shuffled[n_train:])

    train_df = df_in[df_in['student'].isin(train_students)].copy()
    test_df = df_in[df_in['student'].isin(test_students)].copy()

    return train_df, test_df, train_students, test_students


def prepare_features(train_df, test_df):
    vectorizer = create_vectorizer()
    X_train = vectorizer.fit_transform(train_df['tokenized'])
    X_test = vectorizer.transform(test_df['tokenized'])
    y_train = label_encoder.transform(train_df['label'])
    y_test = label_encoder.transform(test_df['label'])
    return X_train, X_test, y_train, y_test


## 7. 병렬 처리 최적화 (Auto-tune)
대표적인 데이터 분할 하나를 사용하여 **최적 병렬 작업 수(n_jobs)를 자동 탐색**합니다.

여러 n_jobs 후보(CPU의 50%, 75%, 전체 등)로 동일한 실험을 실행하고,
가장 빠른 실행 시간을 보인 n_jobs를 이후 실험에 적용합니다.

> 이 자동 튜닝은 실험 시작 전 1회만 수행되며, 시스템 환경에 따라 최적값이 달라질 수 있습니다.

In [17]:

# =========================
# Auto tune n_jobs on a representative Part1 split
# =========================
bench_train_qs = all_combos[0]
bench_combo_name = '_'.join(bench_train_qs)
bench_train_df, bench_test_df = split_by_questions(df, bench_train_qs)

X_train_b, X_test_b, y_train_b, y_test_b = prepare_features(bench_train_df, bench_test_df)
bench_meta = {
    'Context': 'Part1_Bench',
    'Combo': bench_combo_name,
    'Train_Qs': ','.join(bench_train_qs),
    'Test_Qs': ','.join([q for q in ALL_QUESTIONS if q not in bench_train_qs]),
    'N_Train_Qs': len(bench_train_qs),
    'Train_N': len(bench_train_df),
    'Test_N': len(bench_test_df),
}

tune_results = []  # 각 n_jobs 후보의 실행 시간 기록
if AUTO_TUNE_N_JOBS:
    tune_specs = specs[:AUTOTUNE_SPEC_LIMIT] if AUTOTUNE_SPEC_LIMIT > 0 else specs
    print(f'Auto-tune 시작: specs={len(tune_specs)}개, folds={AUTOTUNE_FOLDS}')

    for cand in N_JOBS_CANDIDATES:
        t_tune = time.perf_counter()
        _ = run_algorithms_parallel(
            tune_specs,
            X_train_b,
            y_train_b,
            X_test_b,
            y_test_b,
            bench_meta,
            n_jobs=cand,
            backend=PARALLEL_BACKEND,
            n_splits=AUTOTUNE_FOLDS,
        )
        sec = time.perf_counter() - t_tune
        tune_results.append({'n_jobs': cand, 'sec': sec})
        print(f'  n_jobs={cand:>2}: {sec:.2f}초')

    tune_df = pd.DataFrame(tune_results).sort_values('sec').reset_index(drop=True)
    PARALLEL_N_JOBS = int(tune_df.loc[0, 'n_jobs'])

    tune_csv = SS_OUTPUT_DIR / 'autotune_n_jobs_results.csv'
    tune_df.to_csv(tune_csv, index=False)
    print(f'Auto-tune 결과 저장: {tune_csv}')
    print(f'선택된 PARALLEL_N_JOBS: {PARALLEL_N_JOBS}')


Auto-tune 시작: specs=12개, folds=5
  n_jobs= 6: 17.19초
  n_jobs= 9: 16.15초
  n_jobs=10: 15.72초
  n_jobs=11: 15.75초
  n_jobs=12: 15.65초
  n_jobs=24: 15.54초
Auto-tune 결과 저장: student_split_results_parallel24/autotune_n_jobs_results.csv
선택된 PARALLEL_N_JOBS: 24


## 8. Part 1 실행: 문항 조합 일반화 실험
254개 문항 조합 각각에 대해 24개 알고리즘을 **병렬로 동시 실행**합니다.

예를 들어:
- 학습: Q1, Q3, Q5 (3개 문항의 피드백) → 테스트: Q2, Q4, Q6, Q7, Q8
- 이 과정을 254개 조합 x 24개 알고리즘 = 6,096회 반복

**병렬 처리**: 각 조합 내에서 24개 알고리즘이 `joblib.Parallel`로 동시 실행됩니다.
이를 통해 순차 실행 대비 약 10배 이상의 속도 향상을 얻습니다.

In [18]:

# =========================
# Part 1: Question Combo (Parallel 24 per combo)
# =========================
qc_all_results = []
qc_error_rows = []

total_exp_qc = len(all_combos) * len(specs)
print(f'Part1 전체 실험: {len(all_combos)} 조합 x {len(specs)} 알고리즘 = {total_exp_qc}회')
print(f'Part1 시작: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

pbar = tqdm(total=total_exp_qc, desc='Part1 Combo x 24 Parallel')

for combo_idx, train_qs in enumerate(all_combos):
    combo_name = '_'.join(train_qs)
    test_qs = [q for q in ALL_QUESTIONS if q not in train_qs]

    train_df, test_df = split_by_questions(df, train_qs)
    X_train, X_test, y_train, y_test = prepare_features(train_df, test_df)

    combo_meta = {
        'Combo': combo_name,
        'Train_Qs': ','.join(train_qs),
        'Test_Qs': ','.join(test_qs),
        'N_Train_Qs': len(train_qs),
        'Train_N': len(train_df),
        'Test_N': len(test_df),
    }

    pred_df = test_df[['id', 'feedback_text', 'label']].copy()
    pred_df = pred_df.rename(columns={'label': 'true_label'})

    # 24개 알고리즘을 병렬로 동시 실행 (각 조합당 1회)
    outputs = run_algorithms_parallel(
        specs,
        X_train,
        y_train,
        X_test,
        y_test,
        combo_meta,
        n_jobs=PARALLEL_N_JOBS,
        backend=PARALLEL_BACKEND,
        n_splits=N_FOLDS,
    )

    for out in outputs:
        if out['ok']:
            qc_all_results.append(out['row'])
            pred_df[out['pred_col']] = label_encoder.inverse_transform(out['y_pred'])
        else:
            qc_error_rows.append({'Combo': combo_name, 'Model': out['model'], 'Error': out['error']})

    pred_df.to_excel(QC_PRED_DIR / f'predictions_{combo_name}.xlsx', index=False)

    pbar.update(len(specs))
    pbar.set_postfix(combo=combo_name[:20], errors=len(qc_error_rows))

pbar.close()

qc_results_df = pd.DataFrame(qc_all_results)
qc_csv_path = QC_OUTPUT_DIR / 'qc_all_results_question_combo_parallel24.csv'
qc_results_df.to_csv(qc_csv_path, index=False)

qc_err_path = QC_OUTPUT_DIR / 'qc_errors_parallel24.csv'
pd.DataFrame(qc_error_rows).to_csv(qc_err_path, index=False)

print(f'Part1 결과 저장: {qc_csv_path} ({qc_results_df.shape})')
print(f'Part1 오류 저장: {qc_err_path} ({len(qc_error_rows)}건)')


Part1 전체 실험: 254 조합 x 24 알고리즘 = 6096회
Part1 시작: 2026-03-04 00:49:42


Part1 Combo x 24 Parallel:   0%|          | 0/6096 [00:00<?, ?it/s]

Part1 결과 저장: question_combo_results_parallel24/qc_all_results_question_combo_parallel24.csv ((6096, 20))
Part1 오류 저장: question_combo_results_parallel24/qc_errors_parallel24.csv (0건)


## 9. Part 2 실행: 학생 분할 일반화 실험
학생 단위로 데이터를 분할하여 **새로운 학생의 피드백을 예측하는 능력**을 테스트합니다.

| 비율 | 학습 학생 | 테스트 학생 | 의미 |
|------|----------|-----------|------|
| 80:20 | ~44명 | ~11명 | 충분한 학습 데이터 |
| 70:30 | ~39명 | ~16명 | 일반적 분할 |
| 60:40 | ~33명 | ~22명 | 균형 분할 |
| 50:50 | ~28명 | ~27명 | 동등 분할 |
| 40:60 | ~22명 | ~33명 | 소량 학습, 대량 테스트 |

각 비율에서 3개 seed로 학생을 무작위 배정하고, 24개 알고리즘을 병렬 실행합니다.

In [ ]:

# =========================
# Part 2: Student Split (Parallel 24 per split)
# =========================
splits = {}
splits_info = {}

for train_pct, test_pct in SPLIT_RATIOS:
    ratio_name = f'{train_pct}:{test_pct}'
    for seed in SPLIT_SEEDS:
        split_name = f'{ratio_name}_seed{seed}'
        train_df, test_df, train_students, test_students = split_by_students(df, train_pct, seed)

        splits[split_name] = (train_df, test_df)
        splits_info[split_name] = {
            'train_students': train_students,
            'test_students': test_students,
            'train_n': len(train_df),
            'test_n': len(test_df),
        }

with open(SS_OUTPUT_DIR / 'split_info_parallel24.json', 'w', encoding='utf-8') as f:
    json.dump({
        k: {
            'train_students': sorted(list(v['train_students'])),
            'test_students': sorted(list(v['test_students'])),
            'train_n': v['train_n'],
            'test_n': v['test_n'],
        }
        for k, v in splits_info.items()
    }, f, indent=2, ensure_ascii=False)

ss_all_results = []
ss_error_rows = []

split_items = list(splits.items())
total_exp_ss = len(split_items) * len(specs)
print(f'Part2 전체 실험: {len(split_items)} split x {len(specs)} 알고리즘 = {total_exp_ss}회')
print(f'Part2 시작: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

pbar = tqdm(total=total_exp_ss, desc='Part2 Split x 24 Parallel')

for split_name, (train_df, test_df) in split_items:
    ratio_str, seed_str = split_name.split('_seed')
    split_info = splits_info[split_name]

    split_meta = {
        'Split': split_name,
        'Ratio': ratio_str,
        'Train_Pct': int(ratio_str.split(':')[0]),
        'Seed': int(seed_str),
        'Train_Students': len(split_info['train_students']),
        'Test_Students': len(split_info['test_students']),
        'Train_N': split_info['train_n'],
        'Test_N': split_info['test_n'],
    }

    X_train, X_test, y_train, y_test = prepare_features(train_df, test_df)

    pred_df = test_df[['id', 'feedback_text', 'label']].copy()
    pred_df = pred_df.rename(columns={'label': 'true_label'})

    # 24개 알고리즘을 병렬로 동시 실행 (각 분할당 1회)
    outputs = run_algorithms_parallel(
        specs,
        X_train,
        y_train,
        X_test,
        y_test,
        split_meta,
        n_jobs=PARALLEL_N_JOBS,
        backend=PARALLEL_BACKEND,
        n_splits=N_FOLDS,
    )

    for out in outputs:
        if out['ok']:
            ss_all_results.append(out['row'])
            pred_df[out['pred_col']] = label_encoder.inverse_transform(out['y_pred'])
        else:
            ss_error_rows.append({'Split': split_name, 'Model': out['model'], 'Error': out['error']})

    safe_name = split_name.replace(':', '_')
    pred_df.to_excel(SS_PRED_DIR / f'predictions_{safe_name}.xlsx', index=False)

    pbar.update(len(specs))
    pbar.set_postfix(split=split_name, errors=len(ss_error_rows))

pbar.close()

ss_results_df = pd.DataFrame(ss_all_results)
ss_csv_path = SS_OUTPUT_DIR / 'ss_all_results_student_split_parallel24.csv'
ss_results_df.to_csv(ss_csv_path, index=False)

ss_err_path = SS_OUTPUT_DIR / 'ss_errors_parallel24.csv'
pd.DataFrame(ss_error_rows).to_csv(ss_err_path, index=False)

print(f'Part2 결과 저장: {ss_csv_path} ({ss_results_df.shape})')
print(f'Part2 오류 저장: {ss_err_path} ({len(ss_error_rows)}건)')


## 10. 결과 요약
Part 1과 Part 2의 최고 성능을 요약합니다.

- Part 1에서는 문항 수가 증가할수록 성능이 향상되는 경향을 확인합니다.
- Part 2에서는 학습 데이터 비율과 성능의 관계를 확인합니다.
- 실험 설정은 JSON 파일로 저장되어 재현 가능합니다.

In [19]:

# =========================
# Summary + config
# =========================
if not qc_results_df.empty:
    print()
    print('=== Part1 문항 수별 최고 성능 (Test Kappa) ===')
    for k in range(1, 8):
        subset = qc_results_df[qc_results_df['N_Train_Qs'] == k]
        if len(subset) > 0:
            best = subset.loc[subset['Test_Kappa'].idxmax()]
            print(f"  {k}문항: {best['Combo']} | {best['Model']} | Kappa={best['Test_Kappa']:.4f} F1={best['Test_F1']:.4f}")

if not ss_results_df.empty:
    print()
    print('=== Part2 비율별 최고 성능 (Test Kappa) ===')
    for ratio in [f'{r[0]}:{r[1]}' for r in SPLIT_RATIOS]:
        subset = ss_results_df[ss_results_df['Ratio'] == ratio]
        if len(subset) > 0:
            best = subset.loc[subset['Test_Kappa'].idxmax()]
            print(f"  {ratio}: {best['Model']} | Seed={best['Seed']} | Kappa={best['Test_Kappa']:.4f} F1={best['Test_F1']:.4f}")

config = {
    'Date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'Experiment': 'Question Combo + Student Split ML Parallel24',
    'Random_Seed': RANDOM_SEED,
    'CV_Folds': N_FOLDS,
    'CPU_Logical': CPU_LOGICAL,
    'Parallel_Backend': PARALLEL_BACKEND,
    'Parallel_n_jobs': PARALLEL_N_JOBS,
    'Internal_Model_n_jobs': INTERNAL_MODEL_N_JOBS,
    'AutoTune_Enabled': AUTO_TUNE_N_JOBS,
    'AutoTune_Candidates': N_JOBS_CANDIDATES,
    'AutoTune_Trials': tune_results if 'tune_results' in globals() else [],
    'Part1_Total_Combos': len(all_combos),
    'Part1_Total_Experiments': len(qc_all_results),
    'Part2_Total_Splits': len(splits),
    'Part2_Total_Experiments': len(ss_all_results),
    'Test_Prediction_Method': 'Majority voting from 10-fold CV models',
}

config_path = SS_OUTPUT_DIR / 'experiment_config_parallel24.json'
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print()
print(f'설정 저장: {config_path}')
print(f'완료: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')



=== Part1 문항 수별 최고 성능 (Test Kappa) ===
  1문항: Q2 | C3_Optimal3_SoftVoting | Kappa=0.6707 F1=0.5260
  2문항: Q1_Q8 | C1_Optimal5_SoftVoting | Kappa=0.7270 F1=0.5869
  3문항: Q1_Q4_Q7 | C3_Optimal3_SoftVoting | Kappa=0.7589 F1=0.5978
  4문항: Q1_Q3_Q4_Q7 | C3_Optimal3_SoftVoting | Kappa=0.7798 F1=0.6326
  5문항: Q1_Q2_Q3_Q4_Q7 | C3_Optimal3_SoftVoting | Kappa=0.8073 F1=0.6008
  6문항: Q1_Q2_Q3_Q4_Q6_Q7 | GB | Kappa=0.8357 F1=0.5740
  7문항: Q1_Q2_Q3_Q4_Q6_Q7_Q8 | GB | Kappa=0.8500 F1=0.5733

=== Part2 비율별 최고 성능 (Test Kappa) ===
  80:20: XGB | Seed=456 | Kappa=0.7764 F1=0.5896
  70:30: XGB | Seed=456 | Kappa=0.7932 F1=0.6316
  60:40: C3_Optimal3_SoftVoting | Seed=42 | Kappa=0.7391 F1=0.5996
  50:50: XGB | Seed=42 | Kappa=0.7485 F1=0.5851
  40:60: XGB | Seed=42 | Kappa=0.7375 F1=0.6338

설정 저장: student_split_results_parallel24/experiment_config_parallel24.json
완료: 2026-03-04 08:13:14
